In [15]:
import pandas as pd
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder.appName("Week5").getOrCreate()

# Read Excel file
pdf = pd.read_excel("sales_data.xlsx")

# Convert Pandas DataFrame to Spark DataFrame
df = spark.createDataFrame(pdf)

df_clean = df.dropDuplicates(["user_id", "transaction_date"])

df_clean.show()

print("Original Records :", df.count())
print("After Removing Duplicates :", df_clean.count())

from pyspark.sql.functions import avg

result = (df.filter(df.region == "West")
            .groupBy("product_category")
            .agg(avg("sale_amount").alias("Average_Sales")))

result.show()

df_fill = df.na.fill({
    "status": "Unknown"
})

df_fill.select("status").show(10)

result = df.filter(
    (df.age.between(18,30)) &
    (df.subscription == "Premium")
)

result.show()

print("Total Premium Users :", result.count())

from pyspark.sql.types import TimestampType
from pyspark.sql.functions import col

df_time = (df.withColumn(
            "event_time",
            col("raw_timestamp").cast(TimestampType()))
            .drop("raw_timestamp"))

df_time.printSchema()

df_time.show(5)

clean_df = df.filter(
    (df.email.isNotNull()) &
    (df.username != "")
)

clean_df.show()

print("Remaining Records :", clean_df.count())

from pyspark.sql.functions import min, max, avg

df.agg(
    min("price").alias("Minimum"),
    max("price").alias("Maximum"),
    avg("price").alias("Average")
).show()

from pyspark.sql.functions import sum

result = (
    df
    .dropDuplicates()
    .na.fill({"price":0})
    .groupBy("store_id")
    .agg(sum("price").alias("Total_Revenue"))
)

result.show()

+--------+-------------+
|store_id|Total_Revenue|
+--------+-------------+
|    S105|      52729.0|
|    S102|      58439.0|
|    S104|      60013.0|
|    S101|      70666.0|
|    S103|      59047.0|
+--------+-------------+

